# Phase-1 (concept encoder) — có tốt & đã hội tụ chưa? Có nên FREEZE không?

**Mục tiêu.** Train encoder concept-supervised (Stage-1 = `phase3/pretrain_concept.py`) trên toàn bộ
corpus + tất cả nhãn concept, rồi **đóng băng** nó và đo chất lượng feature cho tác vụ nhị phân
**bệnh / không-bệnh** bằng **PPV@90R** — để trả lời 3 câu hỏi TÁCH BIỆT nhau:

| Câu hỏi | Đo bằng | Ở block |
|---|---|---|
| Đã **hội tụ** chưa? | đường cong train/val loss + val concept-AUROC theo epoch | **B** |
| Encoder có **thực sự tốt** không? | frozen linear-probe PPV@90R trên **train (in-dist)** + **val** + **LOCO chéo-center**, so với SSL thô | **C** |
| Có nên **FREEZE** để tăng generalization? | LOCO PPV@90R của frozen-probe so với unfreeze-finetune | **D** |

**Cảnh báo phương pháp (đọc kỹ).**
- Đo trên **đúng frame đã train** chỉ cho biết *độ khớp / memorization* — luôn lạc quan, **không** chứng minh hội tụ,
  **không** chứng minh generalization. Vì vậy notebook đặt **train (in-dist) cạnh val cạnh LOCO chéo-center**:
  khoảng cách giữa chúng mới là bằng chứng thật.
- Con số quyết định "có nên freeze để tăng generalization" là **LOCO chéo-center PPV@90R**, không phải PPV trên train.
- Log của bạn (`docs/RARE26_EXPERIMENTS.md §D`) đã ghi concept-pretraining **~null so với SSL → retired**. Notebook này là
  **re-test có kiểm soát**: nếu frozen Phase-1 thắng SSL thô ở LOCO (ngoài CI) thì mới là bằng chứng đảo ngược.

**Sự thật về Phase-1 (từ code):** không train nhãn "suspicious" (`overall_suspicion` bị drop, 3.3% quan sát);
train trên corpus concept gộp (KHÔNG có holdout); checkpoint **chỉ lưu backbone, vứt bỏ concept head**.

## 0 · Setup Colab: clone repo + Drive + giải nén `out/` (300K) + weights
Theo đúng pipeline hiện tại của bạn (`phase3/colab_full_pipeline.ipynb`): data 300K nằm ở các **zip chunk trên Drive**,
giải nén vào `out/<dir>/images|labels`. `concept_targets.npz` build từ **toàn bộ out/** = corpus 300K thật.
Không dùng `rare26_concept.tar.gz`.
> Chỉnh `REPO_URL`, `DRIVE_DIR` cho khớp Drive của bạn. Nếu `dinov3` gated: đặt `HF_TOKEN` trong Colab Secrets.

In [ ]:
# ---- clone repo + deps ----
import os, sys, glob, zipfile, shutil, subprocess
REPO_URL = "https://github.com/HuynhDoTanThanh/VISCERA.git"   # <-- repo của bạn (khớp origin)
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system("cd /content && rm -rf rare && git clone -q %s rare" % REPO_URL)
    ROOT = "/content/rare"
    # timm PHẢI đủ mới để có 'vit_base_patch16_dinov3' (>=1.0.16). KHÔNG pin bản cũ.
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","timm","scikit-learn","matplotlib","pandas"])
    import timm; print("timm", timm.__version__, "| dinov3 known:", timm.is_model("vit_base_patch16_dinov3"))
    try:
        from google.colab import userdata
        os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))   # cho dinov3 gated (Colab Secrets)
    except Exception: pass
else:
    ROOT = "/Volumes/Shin/RARE2026"
os.chdir(ROOT); sys.path.insert(0, ROOT); print("ROOT =", ROOT)

# ---- Drive: weights + giải nén các zip chunk vào out/ ----
DRIVE_DIR = "/content/drive/MyDrive/RARE_LG"   # <-- thư mục Drive của bạn
BACKBONES = ["dinov2", "dinov3"]               # đánh giá CẢ HAI backbone
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    # backbone weights
    if os.path.exists(f"{DRIVE_DIR}/dinov2.pth"): shutil.copy(f"{DRIVE_DIR}/dinov2.pth","dinov2.pth"); print("dinov2.pth OK")
    if os.path.exists(f"{DRIVE_DIR}/dinov3.pth"):
        shutil.copy(f"{DRIVE_DIR}/dinov3.pth","dinov3.pth"); print("dinov3.pth OK (Drive)")
    elif "dinov3" in BACKBONES:
        import timm, torch
        m=timm.create_model("vit_base_patch16_dinov3",pretrained=True,img_size=448,num_classes=0)
        torch.save(m.state_dict(),"dinov3.pth"); shutil.copy("dinov3.pth",f"{DRIVE_DIR}/dinov3.pth"); print("downloaded dinov3.pth")
    # giải nén out/ (bỏ qua zip 'dataset')
    os.makedirs("out", exist_ok=True)
    def extract_chunk(zpath):
        with zipfile.ZipFile(zpath) as zf:
            tops={p.split("/")[0] for p in zf.namelist() if p.strip("/")}
            tgt=f"out/{os.path.splitext(os.path.basename(zpath))[0]}" if ("images" in tops or "labels" in tops) else ("." if tops=={"out"} else "out")
            os.makedirs(tgt,exist_ok=True); zf.extractall(tgt)
    for z in [p for p in sorted(glob.glob(f"{DRIVE_DIR}/*.zip")) if "dataset" not in os.path.basename(p).lower()]:
        extract_chunk(z); print("extracted", os.path.basename(z))
print("out dirs:", len(glob.glob("out/*/labels")), "| train labels:", len(glob.glob("out/train/labels/*.json")),
      "| total corpus labels:", len(glob.glob("out/[0-9]*/labels/*.json")))

### 0b · CSV nhị phân (train/val) từ `out/` + ma trận concept 300K + CONFIG

In [ ]:
import os, csv, glob, json, shutil, numpy as np, torch
# ---- train_colab.csv / val_colab.csv (img = out/<split>/images/<name>.jpg) ----
def build_csv(split):
    rows=[]
    for j in glob.glob(f"out/{split}/labels/*.json"):
        d=json.load(open(j))
        if int(d.get("label",-1))<0: continue
        nm=d.get("name",os.path.splitext(os.path.basename(j))[0]); img=f"out/{split}/images/{nm}.jpg"
        if os.path.exists(img): rows.append({"path":img,"center":d.get("center",""),"class":"","label":int(d["label"])})
    with open(f"{split}_colab.csv","w",newline="") as f:
        w=csv.DictWriter(f,fieldnames=["path","center","class","label"]); w.writeheader(); w.writerows(rows)
    print(f"{split}_colab.csv n={len(rows)} pos={sum(r['label'] for r in rows)} centers={sorted({r['center'] for r in rows})}")
    return f"{ROOT}/{split}_colab.csv"
TRAIN_CSV = build_csv("train")
try: VAL_CSV = build_csv("val")
except Exception as e: VAL_CSV=f"{ROOT}/dataset/val.csv"; print("val build skip:",e)

# ---- concept_targets.npz = TOÀN BỘ out/ (300K). Reuse Drive nếu có. ----
os.makedirs("phase3/cache",exist_ok=True)
CT="phase3/cache/concept_targets.npz"
if os.path.exists(CT): print("concept_targets.npz có sẵn.")
elif IN_COLAB and os.path.exists(f"{DRIVE_DIR}/concept_targets.npz"):
    shutil.copy(f"{DRIVE_DIR}/concept_targets.npz",CT); print("REUSED concept_targets.npz (Drive)")
else:
    os.system(f"{sys.executable} -m phase3.build_concept_targets --out {CT}")
    if IN_COLAB: shutil.copy(CT,f"{DRIVE_DIR}/concept_targets.npz")
_n=len(np.load(CT,allow_pickle=True)["paths"]); print(f">>> corpus Phase-1 (300K target): N={_n} frame")

# ---- CONFIG ----
CFG = dict(
    CONCEPT_TARGETS=f"{ROOT}/{CT}", IMG_ROOT=ROOT,     # path trong npz dạng 'out/..' (tương đối ROOT)
    TRAIN_CSV=TRAIN_CSV, VAL_CSV=VAL_CSV,
    OUT_DIR=f"{ROOT}/phase3/cache/phase1_eval", DRIVE_DIR=DRIVE_DIR, BACKBONES=BACKBONES,
    # Stage-1 recipe = ĐÚNG cấu hình ship (colab_full_pipeline cell 8): unfreeze 3, 30 epoch, bs 128
    RETRAIN=True, FORCE_RETRAIN=False,   # block B retrain-có-holdout. FORCE=True để train lại backbone đã có ckpt.
    INIT_CKPT={},          # ckpt ship có sẵn: {"dinov2":".../concept_encoder_dinov2.pt", ...}; để trống = auto tìm Drive
    EPOCHS=30, BS=128, UNFREEZE=3, LR=1e-4, WD=0.05, L2SP=1.0, GRL=1.0,
    DISCRIM="full15", CONTEXT_ROUTE="detach", CERTAIN_FLOOR=0.7, SMOOTH_EPS=0.05, UNC_W=0.1, POS_WEIGHT_CAP=10.0,
    HOLDOUT_FRAC=0.15,
    LIMIT=0,               # >0 = smoke (vd 8000 frame) để xem nhanh dạng đường cong; 0 = full 300K
    WORKERS=8, SEED=0,
)
os.makedirs(CFG["OUT_DIR"],exist_ok=True)
DEV="cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device =",DEV,"| RETRAIN",CFG["RETRAIN"],"| EPOCHS",CFG["EPOCHS"],"| LIMIT",CFG["LIMIT"])

## A · Sự thật về Phase-1 (không train gì — chỉ đọc data)
Xác nhận: (1) `overall_suspicion` có bị drop không, (2) concept nào là dead-const, (3) routing MAIN/CENTER/AUX,
(4) nguồn data & phân bố nhãn. Trả lời trực tiếp "train với suspicious?" và "train trên tập nào?".

In [ ]:
import numpy as np
from phase3.dataset import CONCEPT_NAMES, CONCEPT_ROLE
from phase3.pretrain_concept import route_concepts, priors_and_posw

D = np.load(CFG["CONCEPT_TARGETS"], allow_pickle=True)
paths, value, sup = D["paths"], D["value"], D["supervise"]
print(f"Corpus Phase-1: N = {len(paths)} frame  |  {value.shape[1]} concept  |  KHÔNG có holdout (train hết)")

mi, ci, ai, role_w, dead = route_concepts(value, sup, CFG["DISCRIM"], CFG["CONTEXT_ROUTE"])
prior, posw = priors_and_posw(value, sup, CFG["POS_WEIGHT_CAP"])
nm = lambda idx:[CONCEPT_NAMES[i] for i in idx]
print(f"\nMAIN  (grad→trunk, phân biệt bệnh) [{len(mi)}]: {nm(mi)}")
print(f"CENTER(GRL, đẩy center-cue RA)    [{len(ci)}]: {nm(ci)}")
print(f"AUX   (detached, context)         [{len(ai)}]: {nm(ai)}")
print(f"DROPPED (dead-const / gestalt)    [{len(dead)}]: {dead}")

obs = {CONCEPT_NAMES[i]: float((sup[:,i]>0.5).mean()) for i in range(len(CONCEPT_NAMES))}
trained = set(mi) | set(ci) | set(ai)     # 'trained' = có route vào MỘT head bất kỳ (KHÔNG chỉ dựa vào dead-list)
sj = CONCEPT_NAMES.index("overall_suspicion")
print(f"\n>>> 'overall_suspicion' observed = {obs['overall_suspicion']*100:.1f}%  role={CONCEPT_ROLE['overall_suspicion']}  "
      f"in-MAIN/CENTER/AUX = {sj in trained}")
print(f">>> Kết luận suspicious: {'KHÔNG train — role gestalt không route vào head nào' if sj not in trained else 'CÓ train'}")
print(">>> Tín hiệu phân biệt thực = các concept MAIN ở trên (demarcation, mucosal_irregularity, lesion_present, ...)")

# nguồn data downstream (nhị phân) — TÁCH BIỆT với corpus concept
import csv
from collections import Counter
tr = list(csv.DictReader(open(CFG["TRAIN_CSV"])))
va = [r for r in csv.DictReader(open(CFG["VAL_CSV"])) if r.get("aug","orig")=="orig"]
print(f"\nDownstream binary  train: N={len(tr)}  center={dict(Counter(r['center'] for r in tr))}  "
      f"pos={sum(int(r['label']) for r in tr)}")
print(f"Downstream binary  val  : N={len(va)}  center={dict(Counter(r['center'] for r in va))}  "
      f"pos={sum(int(r['label']) for r in va)}")

## B · Hội tụ? — retrain Stage-1 có holdout + đường cong per-epoch
Vì checkpoint gốc **không lưu lịch sử và không có val split**, đây là cách DUY NHẤT đo hội tụ trung thực:
train lại đúng recipe nhưng giữ 15% frame làm holdout, log mỗi epoch **train loss / val loss / val concept-AUROC (MAIN)**.
**Kết luận:** hội tụ nếu 3 epoch cuối val phẳng và khoảng cách train–val nhỏ; overfit nếu val quay đầu đi lên trong khi train giảm.
> `RETRAIN=False` → bỏ qua, dùng `INIT_CKPT` cho block C/D (nhưng khi đó không có bằng chứng hội tụ).

In [ ]:
import os, json, numpy as np, torch, time
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from phase3.pretrain_concept import (ConceptNet, ConceptDS, masked_concept_loss,
                                     route_concepts, priors_and_posw)

def stratified_holdout(sup, value, main_idx, center, frac, seed):
    # holdout theo center + có/không positive-like để val đại diện
    rng = np.random.default_rng(seed); n = len(value)
    core = value[:, main_idx].max(1) if main_idx else np.zeros(n)
    strata = (core > 0.4).astype(int)
    if center is not None and center.dtype.kind in "US" and (center!="").any():
        cc = np.array([f"{c}_{s}" for c,s in zip(center, strata)])
    else:
        cc = strata.astype(str)
    val = np.zeros(n, bool)
    for g in np.unique(cc):
        idx = np.where(cc==g)[0]; k = max(1, int(len(idx)*frac))
        val[rng.choice(idx, k, replace=False)] = True
    return ~val, val

def train_phase1_with_holdout(backbone):
    D = np.load(CFG["CONCEPT_TARGETS"], allow_pickle=True)
    paths = np.array([os.path.join(CFG["IMG_ROOT"], p) for p in D["paths"]])
    value, sup, center = D["value"], D["supervise"], D.get("center", np.array([""]*len(paths)))
    mi, ci, ai, role_w, dead = route_concepts(value, sup, CFG["DISCRIM"], CFG["CONTEXT_ROUTE"])
    prior, posw = priors_and_posw(value, sup, CFG["POS_WEIGHT_CAP"])
    if CFG["LIMIT"]:
        rng=np.random.default_rng(CFG["SEED"]); sel=rng.choice(len(paths), min(CFG["LIMIT"],len(paths)), replace=False)
        paths,value,sup,center = paths[sel],value[sel],sup[sel],center[sel]
    trm, vam = stratified_holdout(sup, value, mi, center, CFG["HOLDOUT_FRAC"], CFG["SEED"])
    print(f"[{backbone}] train={trm.sum()} holdout={vam.sum()}  MAIN={len(mi)} concepts")

    dl = DataLoader(ConceptDS(paths[trm], value[trm], sup[trm], train=True), batch_size=CFG["BS"],
                    shuffle=True, num_workers=CFG["WORKERS"], drop_last=True, persistent_workers=CFG["WORKERS"]>0)
    dlv= DataLoader(ConceptDS(paths[vam], value[vam], sup[vam], train=False), batch_size=CFG["BS"],
                    shuffle=False, num_workers=CFG["WORKERS"])
    net = ConceptNet(mi, ci, ai, CFG["UNFREEZE"], backbone=backbone).to(DEV)
    sp_ref = {n:p.detach().clone() for n,p in net.backbone.named_parameters() if p.requires_grad} if CFG["L2SP"]>0 else {}
    opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG["EPOCHS"])
    rw=torch.tensor(role_w,device=DEV); pr=torch.tensor(prior,device=DEV); pw=torch.tensor(posw,device=DEV)
    t_mi=torch.tensor(mi,dtype=torch.long,device=DEV); t_ci=torch.tensor(ci,dtype=torch.long,device=DEV); t_ai=torch.tensor(ai,dtype=torch.long,device=DEV)
    kw=dict(certain_floor=CFG["CERTAIN_FLOOR"], smooth_eps=CFG["SMOOTH_EPS"], unc_w=CFG["UNC_W"])
    hist={"train_loss":[], "val_loss":[], "val_auroc":[]}

    for ep in range(CFG["EPOCHS"]):
        net.train(); tot=n=0; lam=CFG["GRL"]*(2.0/(1.0+np.exp(-10.0*ep/max(CFG["EPOCHS"]-1,1)))-1.0)
        for x,v,s in dl:
            x,v,s=x.to(DEV),v.float().to(DEV),s.float().to(DEV); opt.zero_grad()
            pm,pc,pa = net(x,lam)
            loss = masked_concept_loss(pm,v[:,t_mi],s[:,t_mi],rw[t_mi],pr[t_mi],pw[t_mi],**kw)
            if len(ci): loss=loss+masked_concept_loss(pc,v[:,t_ci],s[:,t_ci],rw[t_ci],pr[t_ci],pw[t_ci],**kw)
            if len(ai): loss=loss+masked_concept_loss(pa,v[:,t_ai],s[:,t_ai],rw[t_ai],pr[t_ai],pw[t_ai],**kw)
            if sp_ref: loss=loss+CFG["L2SP"]*sum(((p-sp_ref[nn])**2).mean() for nn,p in net.backbone.named_parameters() if nn in sp_ref)
            loss.backward(); opt.step(); tot+=float(loss.detach()); n+=1
        sched.step()
        # ---- validation: loss + macro concept-AUROC trên MAIN ----
        net.eval(); vtot=vn=0; P=[]; Yv=[]; Sv=[]
        with torch.no_grad():
            for x,v,s in dlv:
                x,v,s=x.to(DEV),v.float().to(DEV),s.float().to(DEV)
                pm,pc,pa=net(x,0.0)
                vtot+=float(masked_concept_loss(pm,v[:,t_mi],s[:,t_mi],rw[t_mi],pr[t_mi],pw[t_mi],**kw)); vn+=1
                P.append(torch.sigmoid(pm).cpu().numpy()); Yv.append(v[:,t_mi].cpu().numpy()); Sv.append(s[:,t_mi].cpu().numpy())
        P=np.concatenate(P); Yv=np.concatenate(Yv); Sv=np.concatenate(Sv)
        aucs=[]
        for j in range(len(mi)):
            m=Sv[:,j]>0.5; yj=(Yv[m,j]>0.5).astype(int)
            if m.sum()>10 and len(set(yj))>1: aucs.append(roc_auc_score(yj, P[m,j]))
        va_auc=float(np.mean(aucs)) if aucs else float("nan")
        hist["train_loss"].append(tot/n); hist["val_loss"].append(vtot/max(vn,1)); hist["val_auroc"].append(va_auc)
        print(f"[{backbone}] ep{ep+1:02d}/{CFG['EPOCHS']} train={tot/n:.4f} val={vtot/max(vn,1):.4f} val_concept_AUROC(MAIN)={va_auc:.3f}", flush=True)
        torch.save({"backbone":net.backbone.state_dict(),"main_idx":mi,"center_idx":ci,"aux_idx":ai,
                    "cfg":{**{k:CFG[k] for k in ['DISCRIM','CONTEXT_ROUTE','UNFREEZE','EPOCHS']},"backbone":backbone},
                    "epoch":ep+1}, f"{CFG['OUT_DIR']}/enc_{backbone}_ep{ep+1}.pt")
    torch.save({"backbone":net.backbone.state_dict(),"main_idx":mi,"center_idx":ci,"aux_idx":ai,
                "cfg":{"backbone":backbone}}, f"{CFG['OUT_DIR']}/enc_{backbone}_final.pt")
    return hist

HISTS = {}
if CFG["RETRAIN"]:
    for bb in CFG["BACKBONES"]:
        fin=f"{CFG['OUT_DIR']}/enc_{bb}_final.pt"; hj=f"{CFG['OUT_DIR']}/hist_{bb}.json"
        if os.path.exists(fin) and os.path.exists(hj) and not CFG.get("FORCE_RETRAIN",False):
            HISTS[bb]=json.load(open(hj)); print(f"[{bb}] SKIP retrain — đã có {os.path.basename(fin)} (đặt FORCE_RETRAIN=True để train lại)")
        else:
            HISTS[bb]=train_phase1_with_holdout(bb); json.dump(HISTS[bb], open(hj,"w"))
else:
    print("RETRAIN=False → bỏ qua block hội tụ, dùng INIT_CKPT ở block C/D.")

# ---- plot convergence ----
if HISTS:
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    for bb,h in HISTS.items():
        e=range(1,len(h["train_loss"])+1)
        ax[0].plot(e,h["train_loss"],"--",label=f"{bb} train"); ax[0].plot(e,h["val_loss"],"-",label=f"{bb} val")
        ax[1].plot(e,h["val_auroc"],"-o",label=f"{bb} val concept-AUROC")
    ax[0].set_title("Loss (train --- vs val ———)  → hội tụ khi val phẳng + gap nhỏ"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].set_title("Val concept-AUROC (MAIN)  → plateau = học xong concept"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].axhline(0.5,ls=":",c="gray")
    plt.tight_layout(); plt.show()
    for bb,h in HISTS.items():
        tail=h["val_loss"][-3:]; slope=(tail[-1]-tail[0])/max(len(tail)-1,1); gap=h["train_loss"][-1]-h["val_loss"][-1]
        print(f"[{bb}] val-loss slope(3 cuối)={slope:+.4f}  train-val gap={gap:+.4f}  → "
              f"{'HỘI TỤ ✓' if abs(slope)<0.01 and abs(gap)<0.15 else 'CHƯA hội tụ / overfit — xem đường cong'}")

## C · Encoder có tốt không? — frozen linear-probe **PPV@90R** trên train + val + LOCO
Đóng băng encoder, trích feature `[cls ⊕ patch.mean]` (1536-d), fit logistic-probe cho tác vụ **bệnh/không-bệnh**.
Đo 3 chế độ để lộ khoảng cách in-dist → generalization:
- **TRAIN (in-dist):** fit & eval trên train — trần lạc quan (memorization).
- **VAL:** fit trên train, eval trên val — giữ nguyên phân bố nhưng frame mới.
- **LOCO c1→c2 / c2→c1:** fit 1 center, test center kia — **con số quyết định generalization** (đã dự báo đúng leaderboard).

So 4 encoder: **raw dinov2, raw dinov3, Phase-1 dinov2, Phase-1 dinov3**. Encoder "thực sự tốt" ⇔ thắng SSL thô ở **LOCO** (ngoài CI), không chỉ ở TRAIN.

In [ ]:
import os, csv, time, numpy as np, torch
from PIL import Image
import timm, timm.models.vision_transformer as vit_mod
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

MEAN=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1); STD=torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)
RAW = {"dinov2":("vit_base_patch14_reg4_dinov2", f"{ROOT}/dinov2.pth", 448),
       "dinov3":("vit_base_patch16_dinov3",      f"{ROOT}/dinov3.pth", 448)}

def _load_backbone(backbone, init_ckpt=None):
    name, raw_path, img = RAW[backbone]
    m = timm.create_model(name, pretrained=False, img_size=img, num_classes=0)
    sd = torch.load(raw_path, map_location="cpu", weights_only=False)
    if backbone=="dinov2": sd={k[len("backbone."):]:v for k,v in sd["teacher"].items() if k.startswith("backbone.")}
    conv=vit_mod.checkpoint_filter_fn(sd,m); conv.pop("mask_token",None); m.load_state_dict(conv, strict=False)
    if init_ckpt:  # nạp encoder Phase-1 (chỉ backbone) đè lên SSL
        ck=torch.load(init_ckpt, map_location="cpu", weights_only=False)
        bk=vit_mod.checkpoint_filter_fn(ck["backbone"],m); bk.pop("mask_token",None)
        mm,uu=m.load_state_dict(bk, strict=False); assert not mm and not uu, f"mismatch {mm[:3]} {uu[:3]}"
        print(f"   [init] {backbone} ← Phase-1 {os.path.basename(init_ckpt)}")
    return m.to(DEV).eval(), img

def extract(paths, backbone, init_ckpt, tag):
    cf=f"{CFG['OUT_DIR']}/feat_{tag}.npz"
    if os.path.exists(cf): d=np.load(cf); return d["F"]
    m,img=_load_backbone(backbone, init_ckpt); F=np.zeros((len(paths),1536),np.float32); t0=time.time()
    with torch.no_grad():
        for st in range(0,len(paths),32):
            ims=[Image.open(p).convert("RGB").resize((img,img),Image.BILINEAR) for p in paths[st:st+32]]
            xb=torch.stack([torch.from_numpy(np.asarray(im,np.float32).copy()).permute(2,0,1)/255. for im in ims])
            xb=((xb-MEAN)/STD).to(DEV); f=m.forward_features(xb)
            F[st:st+xb.shape[0]]=torch.cat([f[:,0], f[:,5:].mean(1)],-1).float().cpu().numpy()
    print(f"   [feat] {tag} {img}px {len(paths)} imgs ({time.time()-t0:.0f}s)"); np.savez(cf,F=F); return F

# --- load labels ---
def load_csv(path, orig_only=False):
    rows=list(csv.DictReader(open(path)))
    if orig_only: rows=[r for r in rows if r.get("aug","orig")=="orig"]
    P=[r["path"] if os.path.isabs(r["path"]) else os.path.join(ROOT,r["path"]) for r in rows]
    return P, np.array([int(r["label"]) for r in rows]), np.array([r["center"] for r in rows])
Ptr,ytr,ctr = load_csv(CFG["TRAIN_CSV"])
Pva,yva,cva = load_csv(CFG["VAL_CSV"], orig_only=True)

# --- encoders to compare: raw SSL vs Phase-1 (ưu tiên encoder SHIP trên Drive) ---
def phase1_ckpt(bb):
    for c in [CFG["INIT_CKPT"].get(bb),
              f"{CFG['DRIVE_DIR']}/concept_encoder_{bb}.pt",   # encoder ship thật (colab_full_pipeline)
              f"{CFG['OUT_DIR']}/enc_{bb}_final.pt"]:          # hoặc encoder retrain block B
        if c and os.path.exists(c): return c
    return None
ENC = []
for bb in CFG["BACKBONES"]:
    ENC.append((f"raw-{bb}", bb, None))
    ck = phase1_ckpt(bb)
    if ck: ENC.append((f"phase1-{bb}", bb, ck)); print(f"phase1-{bb} ← {ck}")
    else:  print(f"[warn] chưa có ckpt Phase-1 cho {bb} — bật RETRAIN (block B) hoặc set CFG['INIT_CKPT']")

FEATS={}
for tag,bb,ck in ENC:
    FEATS[tag]=dict(tr=extract(Ptr,bb,ck,f"{tag}_train"), va=extract(Pva,bb,ck,f"{tag}_val"))
print("done extract:", list(FEATS))

In [ ]:
# --- metrics (khớp competition estimator) ---
def fpr90(y,s,R=.9):
    P=y.sum(); Nn=(y==0).sum()
    if P==0 or Nn==0: return np.nan
    o=np.argsort(-s,kind="mergesort"); ys=y[o]; tp=np.cumsum(ys); fp=np.cumsum(1-ys); rc=tp/P
    return fp[min(np.searchsorted(rc,R),len(rc)-1)]/Nn
def ppv1(f): return np.nan if (f is None or np.isnan(f)) else .009/(.009+.99*f)   # PPV@1% prevalence
def auc(y,s): return roc_auc_score(y,s) if len(set(y))>1 else float("nan")
def boot_ppv(y,s,B=1000):
    rng=np.random.RandomState(0); n=len(y); out=[]
    for _ in range(B):
        idx=rng.randint(0,n,n)
        if y[idx].sum()==0 or (y[idx]==0).sum()==0: continue
        out.append(ppv1(fpr90(y[idx],s[idx])))
    out=np.array([o for o in out if not np.isnan(o)])
    return (round(float(np.median(out)),4), round(float(np.percentile(out,2.5)),4), round(float(np.percentile(out,97.5)),4)) if len(out) else (np.nan,np.nan,np.nan)
def lp(Xtr,ytr,Xte):
    sc=StandardScaler().fit(Xtr); m=LogisticRegression(max_iter=3000,C=1.0,class_weight="balanced").fit(sc.transform(Xtr),ytr)
    return m.predict_proba(sc.transform(Xte))[:,1]

rows=[]
for tag in FEATS:
    Xtr,Xva=FEATS[tag]["tr"],FEATS[tag]["va"]
    # (A) TRAIN in-dist (5-fold để không đọc chính điểm đã fit)
    from sklearn.model_selection import StratifiedKFold
    s_tr=np.zeros(len(ytr))
    for a,b in StratifiedKFold(5,shuffle=True,random_state=0).split(Xtr,ytr):
        s_tr[b]=lp(Xtr[a],ytr[a],Xtr[b])
    # (B) VAL: fit train → eval val
    s_va=lp(Xtr,ytr,Xva)
    # (C) LOCO trên val
    c1=cva=="center_1"; c2=cva=="center_2"
    s_c12=lp(Xva[c1],yva[c1],Xva[c2]); s_c21=lp(Xva[c2],yva[c2],Xva[c1])
    for name,y,s in [("TRAIN(in-dist,5cv)",ytr,s_tr),("VAL",yva,s_va),
                     ("LOCO c1→c2",yva[c2],s_c12),("LOCO c2→c1",yva[c1],s_c21)]:
        f=fpr90(y,s); m,lo,hi=boot_ppv(y,s)
        rows.append(dict(encoder=tag,regime=name,AUROC=round(auc(y,s),4),PPV_at90R=round(ppv1(f),4),
                         FPR_at90R=round(float(f),4) if not np.isnan(f) else None,PPV_bootmed=m,CI=[lo,hi]))

import pandas as pd
df=pd.DataFrame(rows)
pd.set_option("display.width",160,"display.max_rows",100)
print(df.to_string(index=False))
df.to_csv(f"{CFG['OUT_DIR']}/phase1_probe_results.csv",index=False)

## D · Kết luận: encoder tốt không, hội tụ chưa, có nên FREEZE?
Đọc bảng theo 3 câu hỏi:

1. **Tốt?** — so `PPV_bootmed` **LOCO** của `phase1-*` với `raw-*` cùng backbone.
   *Thắng ngoài CI* → concept-pretraining thực sự cải thiện generalization (đảo ngược verdict "retired").
   *Chỉ thắng ở TRAIN mà không ở LOCO* → encoder chỉ memorize, **không** tốt hơn cho center mới.
2. **Hội tụ?** — từ đường cong block B: val-loss phẳng 3 epoch cuối + gap train–val nhỏ.
3. **Freeze?** — so LOCO PPV của frozen-probe (bảng này) với unfreeze-finetune (số D2F trong `docs/RARE26_EXPERIMENTS.md`).
   *frozen ≥ unfreeze (trong CI)* → **đóng băng** encoder (bảo toàn bất biến, ít param, Kumar 2022) là lựa chọn generalization tốt hơn.

Cell dưới in verdict tự động.

In [ ]:
import pandas as pd, numpy as np
df=pd.read_csv(f"{CFG['OUT_DIR']}/phase1_probe_results.csv")
def med(tag,reg):
    r=df[(df.encoder==tag)&(df.regime==reg)]
    return float(r.PPV_bootmed.iloc[0]) if len(r) else np.nan
print("=== VERDICT ===")
for bb in CFG["BACKBONES"]:
    raw,ph=f"raw-{bb}",f"phase1-{bb}"
    if ph not in set(df.encoder): print(f"[{bb}] chưa có ckpt Phase-1 — bỏ qua"); continue
    for leg in ["LOCO c1→c2","LOCO c2→c1"]:
        rv,pv=med(raw,leg),med(ph,leg)
        verdict = "Phase-1 TỐT HƠN ✓" if pv>rv else ("HÒA/kém hơn — không đáng freeze" )
        print(f"[{bb}] {leg}: raw={rv:.4f}  phase1={pv:.4f}  → {verdict}")
    # khoảng cách in-dist → LOCO (đo mức generalization thật)
    tr,lo=med(ph,"TRAIN(in-dist,5cv)"), max(med(ph,"LOCO c1→c2"),med(ph,"LOCO c2→c1"))
    print(f"[{bb}] Phase-1 gap TRAIN {tr:.3f} → LOCO {lo:.3f} = {tr-lo:.3f}  (gap lớn = encoder memorize, không generalize)")
print("\nNhắc: quyết định freeze dựa trên cột LOCO, KHÔNG dựa trên TRAIN. "
      "Nếu Phase-1 không thắng SSL ở LOCO thì verdict 'retired' trong docs được xác nhận lại.")